# Conditional Stable Diffusion (CSD) Verification



In [ ]:
# Import libraries
import numpy as np
import nibabel as nib
import SimpleITK as sitk
import os
import glob
from tqdm import tqdm
import json
from collections import defaultdict

In [ ]:
# Data paths
DATA_DIR = '/net/tscratch/people/plgztabor/ROBUST_PLANNING/DATA'
GENERATED_SAMPLES_DIR = '/net/tscratch/people/plgpiotreksl/generated_samples/images_hu'
OUTPUT_DIR = '/net/tscratch/people/plgpiotreksl/csd_verification_results_fixed'

# Image parameters
SPACING = (1.171875, 1.171875, 3.0)
ORIGIN = (0.0, 0.0, 0.0)
DIRECTION = (-1.0, 0.0, 0.0, 0.0, -1.0, 0.0, 0.0, 0.0, 1.0)
AFF = np.eye(4)

# Structure labels (1=body - skip, 2-5 = structures for analysis)
STRUCTURE_LABELS = [2, 3, 4, 5]
STRUCTURE_NAMES = {
    2: 'rectum',
    3: 'bladder',
    4: 'prostate',
    5: 'femur_heads'
}

# Number of samples per patient
NUM_SAMPLES = 10

# Create output directories
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/probability_maps', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/transforms', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/warped_structures', exist_ok=True)

Używane indeksy sampli (10 z 20): [1, 3, 5, 7, 9, 11, 13, 15, 17, 19]


## Structure Diagnostics

Check which labels are available in the segmentation files.

In [ ]:
# Test patient IDs (manually defined)
test_ids = {'02', '03', '17', '24', '25', '27', '33', '42', '48', '57', '59', '68', '71', '76', '77'}

# Filter only patients for whom we have generated samples
generated_files = glob.glob(f'{GENERATED_SAMPLES_DIR}/Patient_*_sample_*_hu.nii.gz')
available_ids = set([os.path.basename(f).split('_')[1] for f in generated_files])

# Use only patients that are in test_ids AND have generated samples
test_patient_ids = sorted(test_ids & available_ids)
print(f"Test patients ({len(test_patient_ids)}): {test_patient_ids}")

## Registration Functions - Diffeomorphic Demons

In [ ]:
def compute_registration_fixed(ct_frac1_image, generated_image, num_iterations=120):
    """
    Computes Diffeomorphic Demons transform between images.
    
    Direction: fixed=CT_frac1, moving=generated

    Args:
        ct_frac1_image: CT image of fraction 1 (fixed - registration target)
        generated_image: Generated image (moving - to be registered)
        num_iterations: Number of Demons algorithm iterations

    Returns:
        displacement_transform: Transform as DisplacementFieldTransform
        disp_field: Displacement field as image (for saving)
    """
    demons_filter = sitk.DiffeomorphicDemonsRegistrationFilter()
    demons_filter.SetNumberOfIterations(num_iterations)
    demons_filter.SetSmoothDisplacementField(True)
    demons_filter.SetStandardDeviations(0.6)

    # Initial empty displacement field (reference = fixed = CT_frac1)
    initial_displacement_field = sitk.Image(
        ct_frac1_image.GetWidth(), 
        ct_frac1_image.GetHeight(),
        ct_frac1_image.GetDepth(),
        sitk.sitkVectorFloat64)
    initial_displacement_field.CopyInformation(ct_frac1_image)

    # Registration: fixed=CT_frac1, moving=generated
    result_disp = demons_filter.Execute(ct_frac1_image, generated_image, initial_displacement_field)
    displacement_transform = sitk.DisplacementFieldTransform(result_disp)

    # Convert transform to saveable displacement field
    disp_field = sitk.TransformToDisplacementField(
        displacement_transform,
        sitk.sitkVectorFloat64,
        ct_frac1_image.GetSize(),
        ct_frac1_image.GetOrigin(),
        ct_frac1_image.GetSpacing(),
        ct_frac1_image.GetDirection())

    return displacement_transform, disp_field


print("Registration functions loaded.")

## Helper functions for data loading and processing

In [ ]:
def load_nifti_as_sitk(filepath, swap_axes=True):
    """
    Loads a NIfTI file and converts it to a SimpleITK Image.
    
    Args:
        filepath: Path to .nii.gz file
        swap_axes: Whether to swap axes (according to data convention)
    
    Returns:
        sitk.Image with set parameters (origin, spacing, direction)
    """
    data = nib.load(filepath).get_fdata()
    
    if swap_axes:
        data = data.swapaxes(2, 1).swapaxes(1, 0).swapaxes(2, 1)
    
    sitk_img = sitk.GetImageFromArray(data.astype(np.float32))
    sitk_img.SetOrigin(ORIGIN)
    sitk_img.SetSpacing(SPACING)
    sitk_img.SetDirection(DIRECTION)
    
    return sitk_img


def extract_structure_mask(structure_data, label):
    """
    Extracts a binary mask for a given structure label.
    
    Args:
        structure_data: 3D array with segmentation
        label: Structure label number
    
    Returns:
        Binary mask as sitk.Image
    """
    mask = np.zeros(structure_data.shape, dtype=np.float32)
    mask[structure_data == label] = 1.0
    
    sitk_mask = sitk.GetImageFromArray(mask)
    sitk_mask.SetOrigin(ORIGIN)
    sitk_mask.SetSpacing(SPACING)
    sitk_mask.SetDirection(DIRECTION)
    
    return sitk_mask


def apply_transform_to_structure(structure_sitk, displacement_transform, reference_image):
    """
    Applies a transform to a structure image.
    
    Args:
        structure_sitk: Structure image (sitk.Image) - from fraction 1
        displacement_transform: Transform to apply
        reference_image: Reference image (generated)
    
    Returns:
        Warped structure image
    """
    resampler = sitk.ResampleImageFilter()
    resampler.SetReferenceImage(reference_image)
    resampler.SetInterpolator(sitk.sitkLinear)
    resampler.SetDefaultPixelValue(0)
    resampler.SetTransform(displacement_transform)
    
    warped_structure = resampler.Execute(structure_sitk)
    return warped_structure


def compute_probability_map(warped_structures_list):
    """
    Computes a probability map by averaging warped structures.
    
    Args:
        warped_structures_list: List of warped structures (each as numpy array)
    
    Returns:
        Probability map (values from 0 to 1)
    """
    stacked = np.stack(warped_structures_list, axis=0)
    prob_map = np.mean(stacked, axis=0)
    return prob_map


def compute_gt_probability_map(patient_id, structure_label):
    """
    Computes a ground-truth probability map based on all patient fractions.
    
    Args:
        patient_id: Patient ID (e.g. '02')
        structure_label: Structure label number (2-5)
    
    Returns:
        GT probability map
    """
    structure_dir = f'{DATA_DIR}/STRUCTURES/Patient_{patient_id}'
    structure_files = sorted(glob.glob(f'{structure_dir}/Patient_{patient_id}_fraction_*.nii.gz'))
    
    if not structure_files:
        print(f"No structure files for patient {patient_id}")
        return None
    
    structure_masks = []
    for sf in structure_files:
        data = nib.load(sf).get_fdata().swapaxes(2, 1).swapaxes(1, 0).swapaxes(2, 1)
        mask = (data == structure_label).astype(np.float32)
        structure_masks.append(mask)
    
    gt_prob_map = compute_probability_map(structure_masks)
    
    return gt_prob_map


print("Helper functions loaded.")

## Step 1: Compute GT Probability Maps

For each test patient and each structure (except body=1), compute the ground-truth probability map based on all available fractions.

In [ ]:
# Compute GT probability maps for all patients and structures
gt_probability_maps = {}

for patient_id in tqdm(test_patient_ids, desc="Computing GT probability maps"):
    gt_probability_maps[patient_id] = {}
    
    for label in STRUCTURE_LABELS:
        # Check if file already exists
        save_path = f'{OUTPUT_DIR}/probability_maps/GT_Patient_{patient_id}_Structure_{label}_{STRUCTURE_NAMES[label]}.nii.gz'
        
        if os.path.exists(save_path):
            # Load existing map
            gt_prob_map = nib.load(save_path).get_fdata()
            gt_probability_maps[patient_id][label] = gt_prob_map
            continue
        
        # Compute map only if it doesn't exist
        gt_prob_map = compute_gt_probability_map(patient_id, label)
        
        if gt_prob_map is not None:
            gt_probability_maps[patient_id][label] = gt_prob_map
            
            # Save GT probability map
            nifti_img = nib.Nifti1Image(gt_prob_map, affine=AFF)
            nib.save(nifti_img, save_path)

print(f"\nSaved GT probability maps to: {OUTPUT_DIR}/probability_maps/")

## Step 2 & 3: Registration and structure transformation

For each patient:
1. Load fraction 1 CT image (fixed image)
2. Load fraction 1 segmentation
3. For each of the 10 generated images (moving image):
   - Compute transform (Diffeomorphic Demons)
   - Apply transform to fraction 1 segmentation for each structure
4. Average warped segmentations to obtain predicted probability maps

In [ ]:
def process_patient(patient_id, save_intermediate=True):
    """
    Processes a single patient: registration + structure transformation.
    
    Args:
        patient_id: Patient ID (e.g. '02')
        save_intermediate: Whether to save intermediate results (always True for caching)
    
    Returns:
        Dict with predicted probability maps for each structure
    """
    save_intermediate = True
    
    print(f"\n{'='*60}")
    print(f"Processing patient: {patient_id}")
    print(f"{'='*60}")
    
    # Patient data paths
    fraction1_ct_path = f'{DATA_DIR}/CT/Patient_{patient_id}/Patient_{patient_id}_fraction_1_.nii.gz'
    fraction1_struct_path = f'{DATA_DIR}/STRUCTURES/Patient_{patient_id}/Patient_{patient_id}_fraction_1_.nii.gz'
    
    # Check if files exist
    if not os.path.exists(fraction1_ct_path):
        print(f"ERROR: Missing CT file: {fraction1_ct_path}")
        return None
    if not os.path.exists(fraction1_struct_path):
        print(f"ERROR: Missing structure file: {fraction1_struct_path}")
        return None
    
    # Lazy loading - fraction 1 CT will be the fixed image in registration
    ct_frac1_image = None
    struct_data = None
    structure_masks = {}
    
    # Collect all warped structures
    warped_structures = {label: [] for label in STRUCTURE_LABELS}
    
    # Get list of generated samples (use all available)
    sample_files = sorted(glob.glob(f'{GENERATED_SAMPLES_DIR}/Patient_{patient_id}_sample_*_hu.nii.gz'))
    
    print(f"\nFound {len(sample_files)} generated samples")
    
    # Counters for statistics
    cached_count = 0
    computed_count = 0
    
    for sample_idx, sample_path in enumerate(tqdm(sample_files, desc=f"Registering samples")):
        sample_name = os.path.basename(sample_path).replace('.nii.gz', '')
        
        try:
            # Check if all warped structures already exist
            all_warped_exist = True
            warped_paths = {}
            for label in STRUCTURE_LABELS:
                warped_path = f'{OUTPUT_DIR}/warped_structures/warped_{sample_name}_Structure_{label}.nii.gz'
                warped_paths[label] = warped_path
                if not os.path.exists(warped_path):
                    all_warped_exist = False
            
            if all_warped_exist:
                # Load from cache
                for label in STRUCTURE_LABELS:
                    warped_sitk = sitk.ReadImage(warped_paths[label])
                    warped_array = sitk.GetArrayFromImage(warped_sitk)
                    warped_structures[label].append(warped_array)
                cached_count += 1
                continue
            
            # Check cache for transform
            transform_path = f'{OUTPUT_DIR}/transforms/transform_{sample_name}.nii.gz'
            
            if os.path.exists(transform_path):
                # Load transform from disk
                disp_field = sitk.ReadImage(transform_path)
                displacement_transform = sitk.DisplacementFieldTransform(disp_field)
            else:
                # Compute transform: ct_frac1 = fixed, generated = moving
                if ct_frac1_image is None:
                    print("Loading fraction 1 CT image...")
                    ct_frac1_image = load_nifti_as_sitk(fraction1_ct_path)
                
                generated_image = load_nifti_as_sitk(sample_path)
                
                displacement_transform, disp_field = compute_registration_fixed(ct_frac1_image, generated_image)
                
                # Save transform
                os.makedirs(f'{OUTPUT_DIR}/transforms', exist_ok=True)
                sitk.WriteImage(disp_field, transform_path)
                computed_count += 1
            
            # Lazy load structures
            if struct_data is None:
                print("Loading fraction 1 segmentation...")
                struct_data = nib.load(fraction1_struct_path).get_fdata().swapaxes(2, 1).swapaxes(1, 0).swapaxes(2, 1)
                
                for label in STRUCTURE_LABELS:
                    structure_masks[label] = extract_structure_mask(struct_data, label)
                    print(f"  - Structure {label} ({STRUCTURE_NAMES[label]}): "
                          f"sum = {np.sum(sitk.GetArrayFromImage(structure_masks[label])):.0f}")
            
            # Reference = fraction 1 CT (fixed in registration)
            if ct_frac1_image is None:
                ct_frac1_image = load_nifti_as_sitk(fraction1_ct_path)
            
            # Apply transform to each structure
            os.makedirs(f'{OUTPUT_DIR}/warped_structures', exist_ok=True)
            for label in STRUCTURE_LABELS:
                warped_struct = apply_transform_to_structure(
                    structure_masks[label], 
                    displacement_transform, 
                    ct_frac1_image)  # Reference = CT_frac1
                
                warped_array = sitk.GetArrayFromImage(warped_struct)
                warped_structures[label].append(warped_array)
                
                # Save warped structure
                sitk.WriteImage(warped_struct, warped_paths[label])
                    
        except Exception as e:
            print(f"  ERROR for sample {sample_name}: {str(e)}")
            import traceback
            traceback.print_exc()
            continue
    
    print(f"\nStatistics: {cached_count} from cache, {computed_count} computed")
    
    # Compute predicted probability maps
    predicted_prob_maps = {}
    os.makedirs(f'{OUTPUT_DIR}/probability_maps', exist_ok=True)
    
    for label in STRUCTURE_LABELS:
        if len(warped_structures[label]) > 0:
            save_path = f'{OUTPUT_DIR}/probability_maps/PRED_SD_Patient_{patient_id}_Structure_{label}_{STRUCTURE_NAMES[label]}.nii.gz'
            if os.path.exists(save_path):
                # Load from cache
                pred_prob_map = nib.load(save_path).get_fdata()
                predicted_prob_maps[label] = pred_prob_map
                print(f"  Loaded predicted map for structure {label} ({STRUCTURE_NAMES[label]}) from cache")
                continue

            pred_prob_map = compute_probability_map(warped_structures[label])
            predicted_prob_maps[label] = pred_prob_map
            # Save predicted probability map
            nifti_img = nib.Nifti1Image(pred_prob_map, affine=AFF)
            nib.save(nifti_img, save_path)

    return predicted_prob_maps

In [ ]:
# Process all test patients
# Uses caching - if transforms/warped structures already exist, they are loaded from disk.

all_predicted_prob_maps = {}
completed_patients = []

# Check which patients have already been processed
completed_file = f'{OUTPUT_DIR}/completed_patients.json'
if os.path.exists(completed_file):
    with open(completed_file, 'r') as f:
        completed_patients = json.load(f)
    print(f"Found {len(completed_patients)} already processed patients")

for patient_id in test_patient_ids:
    if patient_id in completed_patients:
        print(f"Patient {patient_id} already processed - skipping")
        continue
    
    try:
        pred_maps = process_patient(patient_id)
        
        if pred_maps is not None:
            all_predicted_prob_maps[patient_id] = pred_maps
            completed_patients.append(patient_id)
            
            # Save progress
            with open(completed_file, 'w') as f:
                json.dump(completed_patients, f, indent=4)
                
    except Exception as e:
        print(f"ERROR for patient {patient_id}: {str(e)}")
        import traceback
        traceback.print_exc()
        continue

print(f"\nProcessed {len(completed_patients)} patients")

## Step 4: Metrics - Gray-level Dice and ADICE

Functions for computing metrics comparing GT and predicted probability maps.

In [ ]:
def iou(x, y):
    """
    Computes Intersection over Union (IoU) between two masks.
    """
    dum = x + y
    dum[dum > 0] = 1
    bar = np.copy(x)
    bar[bar > 0] = 1
    foo = np.copy(y)
    foo[foo > 0] = 1
    return np.sum(foo * bar) / np.sum(dum)


def gray_level_dice(gt, pred):
    """
    Computes Gray-level Dice coefficient between two probability maps.
    
    Args:
        gt: Ground truth probability map
        pred: Predicted probability map
    
    Returns:
        Gray-level Dice coefficient
    """
    dice = 2 * np.sum(np.sqrt(gt * pred)) / (np.sum(gt) + np.sum(pred))
    return dice


def adaptive_dice(gt, pred):
    """
    Computes Adaptive Dice (ADICE) - mean Dice across different thresholds.
    
    Args:
        gt: Ground truth probability map
        pred: Predicted probability map
    
    Returns:
        ADICE (mean Dice across thresholds)
    """
    THS = [0.1 + i * 0.1 for i in range(10)]
    adice = []
    
    for TH in THS:
        th_gt = np.zeros(gt.shape, dtype=np.float32)
        th_gt[gt >= TH] = 1
        th_pred = np.zeros(pred.shape, dtype=np.float32)
        th_pred[pred >= TH] = 1
        a = 2 * np.sum(th_gt * th_pred) / (np.sum(th_gt) + np.sum(th_pred))
        adice.append(a)
    
    return np.mean(adice)


def compute_ged(gt_imgs, pred_imgs):
    """
    Computes Generalized Energy Distance (GED).
    
    Args:
        gt_imgs: List of GT images (binary masks)
        pred_imgs: List of predicted images (binary masks)
    
    Returns:
        GED (Generalized Energy Distance)
    """
    # sum1: mean (1-IoU) between all GT-Pred pairs
    sum1 = 0
    for i in range(len(gt_imgs)):
        for j in range(len(pred_imgs)):
            sum1 += 1 - iou(gt_imgs[i], pred_imgs[j])
    sum1 /= (len(gt_imgs) * len(pred_imgs))
    print(f"sum1: {sum1}")
    
    # sum2: mean (1-IoU) within GT
    sum2 = 0
    for i in range(len(gt_imgs) - 1):
        for j in range(i, len(gt_imgs)):
            sum2 += 1 - iou(gt_imgs[i], gt_imgs[j])
    sum2 /= (len(gt_imgs) * (len(gt_imgs) - 1) / 2)
    print(f"sum2: {sum2}")
    
    # sum3: mean (1-IoU) within Pred
    sum3 = 0
    for i in range(len(pred_imgs) - 1):
        for j in range(i, len(pred_imgs)):
            sum3 += 1 - iou(pred_imgs[i], pred_imgs[j])
    sum3 /= (len(pred_imgs) * (len(pred_imgs) - 1) / 2)
    print(f"sum3: {sum3}")
    
    ged = 2*sum1 - sum2 - sum3
    return ged


print("Metric functions loaded.")

In [ ]:
# Compute metrics for all patients and structures
import time

dices = {2: [], 3: [], 4: [], 5: []}
adices = {2: [], 3: [], 4: [], 5: []}
geds = {2: [], 3: [], 4: [], 5: []}

# Load all probability maps from saved files
gt_prob_files = glob.glob(f'{OUTPUT_DIR}/probability_maps/GT_Patient_*_Structure_*.nii.gz')
pred_prob_files = glob.glob(f'{OUTPUT_DIR}/probability_maps/PRED_SD_Patient_*_Structure_*.nii.gz')

print(f"Found {len(gt_prob_files)} GT maps and {len(pred_prob_files)} predicted maps")
print(f"Number of patients: {len(test_patient_ids)}, Number of structures: {len(STRUCTURE_LABELS)}")
print(f"Total combinations to process: {len(test_patient_ids) * len(STRUCTURE_LABELS)}\n")

total_start_time = time.time()
total_combinations = len(STRUCTURE_LABELS) * len(test_patient_ids)
processed_count = 0

for sid in STRUCTURE_LABELS:
    print(f"\n{'='*60}")
    print(f"Structure {sid} ({STRUCTURE_NAMES[sid]})")
    print(f"{'='*60}")
    
    struct_start_time = time.time()
    
    for pid_idx, pid in enumerate(tqdm(test_patient_ids, desc=f"Structure {sid}")):
        patient_start_time = time.time()
        
        gt_path = f'{OUTPUT_DIR}/probability_maps/GT_Patient_{pid}_Structure_{sid}_{STRUCTURE_NAMES[sid]}.nii.gz'
        pred_path = f'{OUTPUT_DIR}/probability_maps/PRED_SD_Patient_{pid}_Structure_{sid}_{STRUCTURE_NAMES[sid]}.nii.gz'
        
        if not os.path.exists(gt_path) or not os.path.exists(pred_path):
            print(f"  Missing files for patient {pid}")
            processed_count += 1
            continue
        
        # Load probability maps
        load_start = time.time()
        gt = nib.load(gt_path).get_fdata()
        pred = nib.load(pred_path).get_fdata()
        load_time = time.time() - load_start
        print(f"  Loaded maps for patient {pid} in {load_time:.1f}s")
        
        # CALCULATE GRAY-LEVEL DICE
        dice_start = time.time()
        dice = gray_level_dice(gt, pred)
        dices[sid].append(float(dice))
        dice_time = time.time() - dice_start
        print(f"    Gray-level DICE: {dice:.4f} (time: {dice_time:.2f}s)")
        
        # CALCULATE ADICE
        adice_start = time.time()
        adice = adaptive_dice(gt, pred)
        adices[sid].append(float(adice))
        adice_time = time.time() - adice_start
        print(f"    ADICE: {adice:.4f} (time: {adice_time:.2f}s)")
        
        # CALCULATE GED - requires individual samples
        ged_start = time.time()
        
        # Load all GT fractions (binary masks)
        structure_dir = f'{DATA_DIR}/STRUCTURES/Patient_{pid}'
        structure_files = sorted(glob.glob(f'{structure_dir}/Patient_{pid}_fraction_*.nii.gz'))
        gt_imgs = []
        for sf in structure_files:
            data = nib.load(sf).get_fdata().swapaxes(2, 1).swapaxes(1, 0).swapaxes(2, 1)
            mask = (data == sid).astype(np.float32)
            gt_imgs.append(mask)
        print(f"    Loaded {len(gt_imgs)} GT fractions for GED")
        
        # Load warped structures (predicted samples) - use all available
        warped_pattern = f'{OUTPUT_DIR}/warped_structures/warped_Patient_{pid}_sample_*_hu_Structure_{sid}.nii.gz'
        warped_files = sorted(glob.glob(warped_pattern))
        
        pred_imgs = []
        for wf in warped_files:
            warped_data = sitk.GetArrayFromImage(sitk.ReadImage(wf))
            binary_mask = (warped_data >= 0.5).astype(np.float32)
            pred_imgs.append(binary_mask)
        print(f"    Loaded {len(pred_imgs)} predicted samples for GED")
        
        # Compute GED if we have enough samples
        if len(gt_imgs) >= 2 and len(pred_imgs) >= 2:
            ged = compute_ged(gt_imgs, pred_imgs)
            geds[sid].append(float(ged))
            ged_str = f"{ged:.4f}"
        else:
            geds[sid].append(None)
            ged_str = "N/A"
            print(f"  Not enough samples for GED (GT: {len(gt_imgs)}, Pred: {len(pred_imgs)})")
        
        ged_time = time.time() - ged_start
        print(f"    GED: {ged_str} (time: {ged_time:.1f}s)")
        patient_time = time.time() - patient_start_time
        
        processed_count += 1
        
        # Estimate remaining time
        elapsed_total = time.time() - total_start_time
        avg_time_per_combo = elapsed_total / processed_count
        remaining_combos = total_combinations - processed_count
        eta_seconds = avg_time_per_combo * remaining_combos
        eta_minutes = eta_seconds / 60
        
        # Detailed logging
        print(f"  Patient {pid}: DICE={dice:.4f}, ADICE={adice:.4f}, GED={ged_str} "
              f"[load:{load_time:.1f}s, dice:{dice_time:.2f}s, adice:{adice_time:.2f}s, ged:{ged_time:.1f}s, "
              f"GT:{len(gt_imgs)}, Pred:{len(pred_imgs)}] "
              f"ETA: {eta_minutes:.1f}min")
        
        # SAVE RESULTS (save after each patient)
        results_path = f'{OUTPUT_DIR}/results.json'
        with open(results_path, 'w') as f:
            json.dump({
                'ids': list(test_patient_ids),
                'dices': {int(k): v for k, v in dices.items()},
                'adices': {int(k): v for k, v in adices.items()},
                'geds': {int(k): v for k, v in geds.items()}
            }, f, indent=4)
    
    # Summary for structure
    struct_time = time.time() - struct_start_time
    valid_geds = [g for g in geds[sid] if g is not None]
    if dices[sid]:
        ged_info = f"GED={np.mean(valid_geds):.4f}+/-{np.std(valid_geds):.4f}" if valid_geds else "GED=N/A"
        print(f"\n  SUMMARY for structure {sid} ({STRUCTURE_NAMES[sid]}):")
        print(f"     DICE={np.mean(dices[sid]):.4f}+/-{np.std(dices[sid]):.4f}")
        print(f"     ADICE={np.mean(adices[sid]):.4f}+/-{np.std(adices[sid]):.4f}")
        print(f"     {ged_info}")
        print(f"     Time: {struct_time:.1f}s ({struct_time/60:.1f}min)")

total_time = time.time() - total_start_time
print("\n" + "=" * 80)
print("FINAL SUMMARY")
print("=" * 80)
print(f"Total time: {total_time:.1f}s ({total_time/60:.1f}min)")

print("\nDICE:")
for key in dices.keys():
    if dices[key]:
        print(f"  {key} ({STRUCTURE_NAMES[key]}): {np.mean(dices[key]):.4f} +/- {np.std(dices[key]):.4f}")

print("\nADICE:")
for key in adices.keys():
    if adices[key]:
        print(f"  {key} ({STRUCTURE_NAMES[key]}): {np.mean(adices[key]):.4f} +/- {np.std(adices[key]):.4f}")

print("\nGED:")
for key in geds.keys():
    valid_geds = [g for g in geds[key] if g is not None]
    if valid_geds:
        print(f"  {key} ({STRUCTURE_NAMES[key]}): {np.mean(valid_geds):.4f} +/- {np.std(valid_geds):.4f}")
    else:
        print(f"  {key} ({STRUCTURE_NAMES[key]}): N/A")

# Final save
results_path = f'{OUTPUT_DIR}/results.json'
with open(results_path, 'w') as f:
    json.dump({
        'ids': list(test_patient_ids),
        'dices': {int(k): v for k, v in dices.items()},
        'adices': {int(k): v for k, v in adices.items()},
        'geds': {int(k): v for k, v in geds.items()}
    }, f, indent=4)

print(f"\nResults saved to: {results_path}")

## Results Summary

Aggregated results for all patients and structures.

In [ ]:
# Load results from file (if running separately)
results_path = f'{OUTPUT_DIR}/results.json'
if os.path.exists(results_path):
    with open(results_path, 'r') as f:
        results = json.load(f)
    
    print("=" * 100)
    print("RESULTS SUMMARY - Conditional Stable Diffusion (CSD)")
    print("=" * 100)
    
    dices = results['dices']
    adices = results['adices']
    geds = results['geds']
    ids = results['ids']
    
    print(f"\nPatients: {ids}")
    print(f"\n{'Structure':<20} {'DICE (mean+/-std)':<25} {'ADICE (mean+/-std)':<25} {'GED (mean+/-std)':<25}")
    print("-" * 100)
    
    for key in ['2', '3', '4', '5']:
        key_int = int(key)
        struct_name = STRUCTURE_NAMES[key_int]
        
        dice_vals = dices.get(key, dices.get(key_int, []))
        adice_vals = adices.get(key, adices.get(key_int, []))
        ged_vals_raw = geds.get(key, geds.get(key_int, []))
        ged_vals = [float(g) for g in ged_vals_raw if g is not None]
        
        if dice_vals:
            dice_str = f"{np.mean(dice_vals):.4f} +/- {np.std(dice_vals):.4f}"
            adice_str = f"{np.mean(adice_vals):.4f} +/- {np.std(adice_vals):.4f}"
            ged_str = f"{np.mean(ged_vals):.4f} +/- {np.std(ged_vals):.4f}" if ged_vals else "N/A"
            
            print(f"{key} ({struct_name}){'':<8} {dice_str:<25} {adice_str:<25} {ged_str:<25}")
    
    print("=" * 100)
    
    # Overall summary
    all_dices = []
    all_adices = []
    all_geds = []
    for key in ['2', '3', '4', '5']:
        all_dices.extend(dices.get(key, dices.get(int(key), [])))
        all_adices.extend(adices.get(key, adices.get(int(key), [])))
        ged_vals_raw = geds.get(key, geds.get(int(key), []))
        all_geds.extend([float(g) for g in ged_vals_raw if g is not None])
    
    if all_dices:
        print(f"\nOVERALL MEAN:")
        print(f"  DICE:  {np.mean(all_dices):.4f} +/- {np.std(all_dices):.4f}")
        print(f"  ADICE: {np.mean(all_adices):.4f} +/- {np.std(all_adices):.4f}")
        if all_geds:
            print(f"  GED:   {np.mean(all_geds):.4f} +/- {np.std(all_geds):.4f}")
else:
    print(f"Results file not found: {results_path}")

## Probability Map Visualization

Comparative visualization of GT vs predicted probability maps for a selected patient and structure.

In [ ]:
import matplotlib.pyplot as plt

def visualize_probability_maps(patient_id, structure_label, slice_idx=None):
    """
    Comparative visualization of GT and predicted probability maps.
    
    Args:
        patient_id: Patient ID
        structure_label: Structure label number
        slice_idx: Slice index (if None, automatically selected)
    """
    gt_path = f'{OUTPUT_DIR}/probability_maps/GT_Patient_{patient_id}_Structure_{structure_label}_{STRUCTURE_NAMES[structure_label]}.nii.gz'
    pred_path = f'{OUTPUT_DIR}/probability_maps/PRED_SD_Patient_{patient_id}_Structure_{structure_label}_{STRUCTURE_NAMES[structure_label]}.nii.gz'
    
    if not os.path.exists(gt_path) or not os.path.exists(pred_path):
        print(f"Missing files for patient {patient_id}, structure {structure_label}")
        return
    
    gt_map = nib.load(gt_path).get_fdata()
    pred_map = nib.load(pred_path).get_fdata()
    
    # Select slice with maximum sum of values
    if slice_idx is None:
        slice_sums = np.sum(gt_map, axis=(0, 1))
        slice_idx = np.argmax(slice_sums)
    
    # Create visualization
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # GT map
    im1 = axes[0].imshow(gt_map[:, :, slice_idx].T, cmap='hot', vmin=0, vmax=1, origin='lower')
    axes[0].set_title(f'GT Probability Map\n(Slice {slice_idx})')
    axes[0].axis('off')
    plt.colorbar(im1, ax=axes[0], fraction=0.046)
    
    # Predicted map
    im2 = axes[1].imshow(pred_map[:, :, slice_idx].T, cmap='hot', vmin=0, vmax=1, origin='lower')
    axes[1].set_title(f'Predicted (SD) Probability Map\n(Slice {slice_idx})')
    axes[1].axis('off')
    plt.colorbar(im2, ax=axes[1], fraction=0.046)
    
    # Difference
    diff = gt_map[:, :, slice_idx] - pred_map[:, :, slice_idx]
    im3 = axes[2].imshow(diff.T, cmap='RdBu', vmin=-0.5, vmax=0.5, origin='lower')
    axes[2].set_title(f'Difference (GT - Pred)\n(Slice {slice_idx})')
    axes[2].axis('off')
    plt.colorbar(im3, ax=axes[2], fraction=0.046)
    
    # Compute metrics
    gld = gray_level_dice(gt_map, pred_map)
    adice = adaptive_dice(gt_map, pred_map)
    
    plt.suptitle(f'Patient {patient_id} - {STRUCTURE_NAMES[structure_label]}\nGLD: {gld:.4f}, ADICE: {adice:.4f}', fontsize=14)
    plt.tight_layout()
    
    # Save figure
    fig_path = f'{OUTPUT_DIR}/figures'
    os.makedirs(fig_path, exist_ok=True)
    plt.savefig(f'{fig_path}/comparison_Patient_{patient_id}_Structure_{structure_label}.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    return fig


# Example visualization for first patient and prostate (label=4)
# visualize_probability_maps(test_patient_ids[0], 4)